# MongoDB 장애 대응 RAG 시스템

LangChain + Voyage AI + MongoDB Atlas + OpenAI를 활용한  
**하이브리드 검색(Hybrid Search) + RRF** 기반 RAG 파이프라인

---

## 사용 기술

| 역할 | 기술 |
|------|------|
| 임베딩 | Voyage AI `voyage-4` (1024차원) |
| 벡터 저장소 | MongoDB Atlas Vector Search |
| 전문 검색 | MongoDB Atlas Search (Full-Text) |
| 검색 융합 | RRF (Reciprocal Rank Fusion) |
| 답변 생성 | OpenAI `gpt-4o-mini` |
| RAG 파이프라인 | LangChain LCEL |

## 아키텍처

```
사용자 질문
     │
     ├─── [벡터 검색] voyage-4 임베딩 → $vectorSearch → 순위 목록 A
     │
     ├─── [전문 검색] $search (Atlas Search) → 순위 목록 B
     │
     └─── [RRF 융합] 1/(k+rank_A) + 1/(k+rank_B) → 최종 순위
                │
                └─── 상위 문서 → GPT-4o-mini → 최종 답변
```

## 사전 준비

1. `pip install -r requirements.txt`
2. Atlas 무료 계정 생성
   - 무료계정 생성 링크 - https://www.mongodb.com/cloud/atlas/register
   - Atlas URI 링크 복사 방법 - https://www.mongodb.com/ko-kr/docs/languages/python/pymongo-driver/current/get-started/#create-a-connection-string
2. MongoDB Atlas에 접속하여 voyage ai api key 생성 (아래 기술문서 부분 참조)
   - https://www.mongodb.com/ko-kr/docs/voyageai/quickstart/?llm-provider=anthropic#create-a-model-api-key
3. `.env` 파일에 API 키 설정 (.env.example 파일을 복사하여 .env 파일명으로 신규 생성 후 API 키 설정 진행)

## Step 0. MongoDB Atlas 인덱스 자동 생성

pymongo 4.7+의 `SearchIndexModel` API를 사용해 **코드에서 직접** 인덱스를 생성합니다.  
Atlas UI에서 수동으로 생성할 필요가 없습니다.

생성되는 인덱스 2종:

| 인덱스 이름 | 타입 | 용도 |
|-------------|------|------|
| `vector_index` | `vectorSearch` | `$vectorSearch` 시맨틱 검색 |
| `search_index` | `search` | `$search` 키워드 전문 검색 |

> **참고:** 인덱스 생성은 비동기로 진행됩니다.  
> `wait_for_indexes_ready()` 함수로 `READY` 상태가 될 때까지 자동으로 기다립니다.

## Step 1. 라이브러리 설치

In [1]:
# 필요한 라이브러리 설치 (pymongo 4.7+ 필요 - SearchIndexModel API)
!pip install "pymongo>=4.7" langchain langchain-core langchain-openai langchain-voyageai langchain-mongodb python-dotenv openai voyageai -q

## Step 2. 환경 변수 설정

In [ ]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()

# 또는 직접 입력
# os.environ["MONGODB_URI"]    = "mongodb+srv://..."
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["VOYAGE_API_KEY"] = "pa-..."

MONGODB_URI    = os.getenv("MONGODB_URI", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
VOYAGE_API_KEY = os.getenv("VOYAGE_API_KEY", "")

# 설정 확인
print("MONGODB_URI    :", "✅ 설정됨" if MONGODB_URI    else "❌ 미설정")
print("OPENAI_API_KEY :", "✅ 설정됨" if OPENAI_API_KEY else "❌ 미설정")
print("VOYAGE_API_KEY :", "✅ 설정됨" if VOYAGE_API_KEY else "❌ 미설정")

MONGODB_URI    : ✅ 설정됨
OPENAI_API_KEY : ✅ 설정됨
VOYAGE_API_KEY : ✅ 설정됨


## Step 3. 설정 상수 정의

In [3]:
# ── MongoDB 설정 ──────────────────────────────────────────────
DB_NAME           = "mongodb_troubleshooting"  # 데이터베이스 이름
COLLECTION_NAME   = "knowledge_base"           # 컬렉션 이름
VECTOR_INDEX_NAME = "vector_index"             # Atlas 벡터 인덱스 이름
SEARCH_INDEX_NAME = "search_index"             # Atlas 전문 검색 인덱스 이름
TEXT_FIELD        = "text"                     # 본문 필드명
EMBEDDING_FIELD   = "embedding"                # 임베딩 필드명

# ── 모델 설정 ─────────────────────────────────────────────────
EMBEDDING_MODEL   = "voyage-4"                 # Voyage AI 임베딩 모델
EMBEDDING_DIMS    = 1024                       # voyage-4 차원 수
LLM_MODEL         = "gpt-4o-mini"             # OpenAI LLM 모델

print(f"임베딩 모델 : {EMBEDDING_MODEL} ({EMBEDDING_DIMS}차원)")
print(f"LLM 모델   : {LLM_MODEL}")
print(f"DB/Collection: {DB_NAME}.{COLLECTION_NAME}")

임베딩 모델 : voyage-4 (1024차원)
LLM 모델   : gpt-4o-mini
DB/Collection: mongodb_troubleshooting.knowledge_base


## Step 4. 지식 베이스 데이터 준비

MongoDB 장애 대응 시나리오 10가지를 지식 베이스로 사용합니다.

In [4]:
from langchain_core.documents import Document
from typing import List, Dict, Any

# MongoDB 장애 대응 지식 베이스
MONGODB_KNOWLEDGE_BASE = [
    {
        "title": "Connection Pool 고갈 문제",
        "category": "connection",
        "source": "MongoDB 연결 관리 가이드",
        "text": (
            "MongoDB Connection Pool 고갈 문제 해결 가이드\n\n"
            "증상:\n"
            "- 'connection pool exhausted' 또는 'too many connections' 오류 발생\n"
            "- 애플리케이션 응답 지연 및 타임아웃\n"
            "- mongostat에서 connections 수가 maxIncomingConnections에 근접\n\n"
            "원인:\n"
            "- 커넥션을 반납하지 않는 코드 (커넥션 누수)\n"
            "- 갑작스러운 트래픽 증가\n"
            "- maxPoolSize 설정이 너무 낮음\n"
            "- 슬로우 쿼리로 인한 커넥션 점유 시간 증가\n\n"
            "해결 방법:\n"
            "1. 현재 커넥션 상태 확인\n"
            "   db.serverStatus().connections\n\n"
            "2. maxPoolSize 조정 (드라이버 설정)\n"
            "   MongoClient(uri, maxPoolSize=100, waitQueueTimeoutMS=5000)\n\n"
            "3. 커넥션 누수 확인\n"
            "   db.currentOp()로 장시간 열려있는 세션 확인\n\n"
            "예방:\n"
            "- 커넥션 풀 모니터링 (Atlas Metrics > Connections)\n"
            "- 알림 설정 (connections > 임계값)\n"
            "- 슬로우 쿼리 최적화로 커넥션 점유 시간 단축"
        ),
    },
    {
        "title": "인덱스 누락으로 인한 슬로우 쿼리",
        "category": "performance",
        "source": "MongoDB 성능 최적화 가이드",
        "text": (
            "인덱스 누락으로 인한 슬로우 쿼리 해결 가이드\n\n"
            "증상:\n"
            "- 쿼리 응답 시간이 수 초 이상\n"
            "- Atlas Profiler에 COLLSCAN(전체 스캔) 표시\n"
            "- CPU 사용률 급상승\n\n"
            "진단 방법:\n"
            "1. 슬로우 쿼리 로그 확인\n"
            "   db.system.profile.find({millis: {$gt: 100}}).sort({ts: -1}).limit(10)\n\n"
            "2. explain()으로 실행 계획 분석\n"
            "   db.collection.find(query).explain('executionStats')\n\n"
            "해결 방법:\n"
            "1. 필요한 인덱스 생성\n"
            "   db.orders.createIndex({user_id: 1, created_at: -1})\n\n"
            "2. 복합 인덱스 설계 원칙 (ESR 규칙)\n"
            "   - Equality(동등 조건) 필드 먼저\n"
            "   - Sort(정렬) 필드 다음\n"
            "   - Range(범위 조건) 필드 마지막\n\n"
            "예방:\n"
            "- 새 쿼리 패턴 도입 시 인덱스 계획 수립\n"
            "- Atlas Performance Advisor 권장 인덱스 모니터링"
        ),
    },
    {
        "title": "Replication Lag 복제 지연 문제",
        "category": "replication",
        "source": "MongoDB 복제 운영 가이드",
        "text": (
            "MongoDB Replication Lag(복제 지연) 해결 가이드\n\n"
            "증상:\n"
            "- rs.printSlaveReplicationInfo()에서 지연 시간이 큰 경우\n"
            "- Secondary에서 읽기 시 오래된 데이터 반환\n\n"
            "원인:\n"
            "- Primary에 과도한 쓰기 부하\n"
            "- Secondary 서버의 리소스 부족 (CPU, I/O)\n"
            "- 네트워크 대역폭 부족\n"
            "- Oplog 크기 부족으로 인한 롤백\n\n"
            "진단 방법:\n"
            "1. 복제 상태 확인\n"
            "   rs.status()\n"
            "   rs.printSlaveReplicationInfo()\n\n"
            "해결 방법:\n"
            "1. 즉각적 조치 - 부하 분산\n"
            "   - 읽기 요청을 Primary로 임시 전환\n"
            "   - 배치 작업 일시 중지\n\n"
            "2. Oplog 크기 증설\n"
            "   mongod.conf: replication.oplogSizeMB: 51200\n\n"
            "예방:\n"
            "- Oplog 크기를 최소 24-72시간 분량으로 설정\n"
            "- Secondary 서버 사양을 Primary와 동일하게 유지"
        ),
    },
    {
        "title": "WiredTiger 캐시 메모리 부족",
        "category": "memory",
        "source": "MongoDB 메모리 관리 가이드",
        "text": (
            "WiredTiger 캐시 메모리 부족 문제 해결 가이드\n\n"
            "증상:\n"
            "- 쿼리 성능 급격히 저하\n"
            "- 디스크 I/O 급증\n"
            "- 'WiredTiger eviction' 경고 메시지\n\n"
            "원인:\n"
            "- 데이터셋이 WiredTiger 캐시 크기 초과\n"
            "- 메모리 설정이 서버 사양에 비해 낮게 설정됨\n\n"
            "해결 방법:\n"
            "1. WiredTiger 캐시 크기 증설\n"
            "   mongod.conf:\n"
            "     storage.wiredTiger.engineConfig.cacheSizeGB: 4\n\n"
            "2. 불필요한 인덱스 제거로 메모리 절약\n\n"
            "예방:\n"
            "- 캐시 히트율 지속 모니터링 (목표: 95% 이상)\n"
            "- Atlas 사용 시 적절한 인스턴스 티어 선택"
        ),
    },
    {
        "title": "Disk Space 디스크 공간 부족 장애",
        "category": "storage",
        "source": "MongoDB 스토리지 관리 가이드",
        "text": (
            "MongoDB Disk Space 부족 장애 해결 가이드\n\n"
            "증상:\n"
            "- 'no space left on device' 오류\n"
            "- 쓰기 작업 실패\n"
            "- MongoDB 프로세스 비정상 종료\n\n"
            "즉각적인 조치:\n"
            "1. 현재 디스크 사용량 확인\n"
            "   df -h  (OS 레벨)\n"
            "   db.stats()  (MongoDB 레벨)\n\n"
            "해결 방법:\n"
            "1. 오래된 데이터 아카이브 또는 삭제\n"
            "   db.logs.deleteMany({createdAt: {$lt: new Date('2024-01-01')}})\n\n"
            "2. compact 명령으로 공간 회수\n"
            "   db.runCommand({compact: 'collection_name'})\n\n"
            "3. TTL 인덱스로 자동 데이터 만료 설정\n"
            "   db.logs.createIndex({createdAt: 1}, {expireAfterSeconds: 2592000})\n\n"
            "예방:\n"
            "- 디스크 사용량 80% 도달 시 알림 설정\n"
            "- Atlas Online Archive 활용"
        ),
    },
    {
        "title": "Lock 경합 (Lock Contention) 문제",
        "category": "locking",
        "source": "MongoDB 락 관리 가이드",
        "text": (
            "MongoDB Lock 경합(Lock Contention) 해결 가이드\n\n"
            "증상:\n"
            "- 전반적인 성능 저하\n"
            "- globalLock.ratio 값이 0.5 이상\n"
            "- 쿼리 대기 시간 증가\n\n"
            "진단 방법:\n"
            "1. 현재 실행 중인 작업 확인\n"
            "   db.currentOp({active: true, waitingForLock: true})\n\n"
            "해결 방법:\n"
            "1. 장시간 실행 중인 작업 중단\n"
            "   db.killOp(opid)\n\n"
            "2. 대용량 작업 배치 처리로 분할\n"
            "   - 한 번에 1000건씩 처리하고 sleep 추가\n\n"
            "예방:\n"
            "- 대용량 작업 시 배치 처리 패턴 적용\n"
            "- 모든 쿼리에 적절한 인덱스 확보"
        ),
    },
    {
        "title": "Primary 선출 실패 (Election Failure)",
        "category": "replication",
        "source": "MongoDB 복제셋 운영 가이드",
        "text": (
            "MongoDB Replica Set Primary 선출 실패 해결 가이드\n\n"
            "증상:\n"
            "- 모든 멤버가 SECONDARY 또는 UNKNOWN 상태\n"
            "- 쓰기 작업 불가 ('not master' 오류)\n"
            "- rs.status()에서 no primary 상태\n\n"
            "원인:\n"
            "- 네트워크 파티션 (스플릿 브레인)\n"
            "- 과반수(Majority) 멤버에 연결 불가\n\n"
            "해결 방법:\n"
            "1. 네트워크 복구 후 자동 선출 대기 (통상 10-30초)\n\n"
            "2. 특정 멤버 우선순위 조정\n"
            "   cfg = rs.conf()\n"
            "   cfg.members[0].priority = 2\n"
            "   rs.reconfig(cfg)\n\n"
            "예방:\n"
            "- 항상 홀수 개의 투표 멤버 유지 (3, 5, 7개)\n"
            "- 멤버를 서로 다른 AZ에 배포"
        ),
    },
    {
        "title": "Atlas Vector Search 인덱스 오류",
        "category": "search",
        "source": "MongoDB Atlas 검색 가이드",
        "text": (
            "MongoDB Atlas Vector Search 인덱스 오류 해결 가이드\n\n"
            "증상:\n"
            "- '$vectorSearch is not allowed' 오류\n"
            "- 검색 결과가 빈 배열 반환\n"
            "- 인덱스 상태가 FAILED 또는 BUILDING\n\n"
            "원인:\n"
            "- Atlas Search 인덱스가 생성되지 않음\n"
            "- 인덱스 이름 불일치\n"
            "- numDimensions가 임베딩 모델 차원수와 불일치\n\n"
            "해결 방법:\n"
            "1. Vector Search 인덱스 설정 확인\n"
            "   numDimensions: 1024  (voyage-4 모델)\n\n"
            "2. 인덱스 이름 코드와 일치 여부 확인\n\n"
            "3. 인덱스 재빌드 대기\n"
            "   - 대용량 컬렉션은 수 분 소요\n\n"
            "예방:\n"
            "- 인덱스 이름을 코드 상수로 관리"
        ),
    },
    {
        "title": "OOM (Out of Memory) 킬러 발생",
        "category": "memory",
        "source": "MongoDB 메모리 관리 가이드",
        "text": (
            "MongoDB OOM (Out of Memory) 킬러 발생 해결 가이드\n\n"
            "증상:\n"
            "- MongoDB 프로세스가 갑자기 종료됨\n"
            "- 'Killed process (mongod)' 로그\n"
            "- 시스템 메모리 사용량 100% 도달\n\n"
            "원인:\n"
            "- WiredTiger 캐시 + 인덱스가 가용 메모리 초과\n"
            "- 다른 프로세스와의 메모리 경쟁\n\n"
            "해결 방법:\n"
            "1. WiredTiger 캐시 크기 제한\n"
            "   cacheSizeGB: 2  (가용 메모리의 50% 이하)\n\n"
            "2. Swap 설정\n"
            "   fallocate -l 4G /swapfile && mkswap /swapfile && swapon /swapfile\n\n"
            "예방:\n"
            "- 메모리 사용률 85% 알림 설정\n"
            "- Atlas 사용 시 적절한 티어 선택"
        ),
    },
    {
        "title": "Mongodump / Mongorestore 백업 및 복구",
        "category": "backup",
        "source": "MongoDB 백업 복구 가이드",
        "text": (
            "MongoDB 백업 및 복구 완전 가이드\n\n"
            "Mongodump (백업):\n"
            "1. 전체 데이터베이스 백업\n"
            "   mongodump --uri='mongodb+srv://...' --out=/backup/$(date +%Y%m%d)\n\n"
            "2. 특정 컬렉션 백업\n"
            "   mongodump --uri='...' --db=mydb --collection=users --out=/backup/\n\n"
            "Mongorestore (복구):\n"
            "1. 전체 복구\n"
            "   mongorestore --uri='mongodb+srv://...' /backup/20240101/\n\n"
            "2. 병렬 복구로 속도 향상\n"
            "   mongorestore --uri='...' --numParallelCollections=4 /backup/\n\n"
            "Atlas 백업:\n"
            "- Atlas > Cluster > Backup > Take Snapshot\n"
            "- Point-in-Time Recovery (PITR) 활용\n"
            "- M10 이상에서 Continuous Cloud Backup 권장\n\n"
            "주의사항:\n"
            "- Replica Set에서 --oplog 옵션으로 일관성 확보\n"
            "- 대용량 백업 시 --gzip 옵션으로 압축"
        ),
    },
]


def prepare_documents(knowledge_base: list) -> List[Document]:
    """지식 베이스를 LangChain Document 리스트로 변환합니다."""
    docs = [
        Document(
            page_content=item["text"],
            metadata={
                "title": item["title"],
                "category": item["category"],
                "source": item["source"],
            },
        )
        for item in knowledge_base
    ]
    print(f"✅ {len(docs)}개 문서 준비 완료")
    for doc in docs:
        print(f"   - [{doc.metadata['category']}] {doc.metadata['title']}")
    return docs


docs = prepare_documents(MONGODB_KNOWLEDGE_BASE)

✅ 10개 문서 준비 완료
   - [connection] Connection Pool 고갈 문제
   - [performance] 인덱스 누락으로 인한 슬로우 쿼리
   - [replication] Replication Lag 복제 지연 문제
   - [memory] WiredTiger 캐시 메모리 부족
   - [storage] Disk Space 디스크 공간 부족 장애
   - [locking] Lock 경합 (Lock Contention) 문제
   - [replication] Primary 선출 실패 (Election Failure)
   - [search] Atlas Vector Search 인덱스 오류
   - [memory] OOM (Out of Memory) 킬러 발생
   - [backup] Mongodump / Mongorestore 백업 및 복구


## Step 5. 컴포넌트 초기화 (임베딩, LLM, MongoDB)

In [5]:
import pymongo
from pymongo import MongoClient
from langchain_voyageai import VoyageAIEmbeddings
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_openai import ChatOpenAI

# ── MongoDB Atlas 연결 ────────────────────────────────────────
client = MongoClient(MONGODB_URI)
client.admin.command("ping")  # 연결 테스트
print("✅ MongoDB Atlas 연결 성공")

db         = client[DB_NAME]
collection = db[COLLECTION_NAME]

# ── Voyage AI 임베딩 ──────────────────────────────────────────
embeddings = VoyageAIEmbeddings(
    voyage_api_key=VOYAGE_API_KEY,
    model=EMBEDDING_MODEL,
)
print(f"✅ Voyage AI 임베딩 초기화 완료 (모델: {EMBEDDING_MODEL})")

# ── OpenAI LLM ───────────────────────────────────────────────
llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0,
    openai_api_key=OPENAI_API_KEY,
)
print(f"✅ OpenAI LLM 초기화 완료 (모델: {LLM_MODEL})")

✅ MongoDB Atlas 연결 성공
✅ Voyage AI 임베딩 초기화 완료 (모델: voyage-4)
✅ OpenAI LLM 초기화 완료 (모델: gpt-4o-mini)


## Step 6. 문서를 MongoDB Atlas에 업로드

> voyage-4로 각 문서의 임베딩을 생성하고 MongoDB에 저장합니다.  
> **처음 실행 시에만** 업로드가 필요합니다. 이후에는 기존 데이터를 재사용합니다.

In [6]:
import time

# force_reload=True 로 설정하면 기존 데이터를 삭제하고 재업로드
FORCE_RELOAD = False

existing_count = collection.count_documents({})

if not FORCE_RELOAD and existing_count > 0:
    print(f"✅ 기존 데이터 사용 중 ({existing_count}개 문서)")
    print("   재업로드하려면 FORCE_RELOAD = True 로 설정하세요.")
else:
    print(f"📥 {len(docs)}개 문서를 MongoDB Atlas에 업로드 중...")
    print("   임베딩 생성 중 (수십 초 소요)...")
    collection.drop()

    MongoDBAtlasVectorSearch.from_documents(
        documents=docs,
        embedding=embeddings,
        collection=collection,
        index_name=VECTOR_INDEX_NAME,
    )
    print(f"✅ 업로드 완료 ({collection.count_documents({})}개 문서)")

# 벡터 스토어 인스턴스 (나중에 활용 가능)
vector_store = MongoDBAtlasVectorSearch(
    collection=collection,
    embedding=embeddings,
    index_name=VECTOR_INDEX_NAME,
    text_key=TEXT_FIELD,
    embedding_key=EMBEDDING_FIELD,
)
print("✅ 벡터 스토어 준비 완료")

# 업로드된 문서 샘플 확인
sample = collection.find_one({}, {"text": 1, "metadata": 1})
if sample:
    print(f"\n[샘플 문서]")
    print(f"  제목: {sample.get('metadata', {}).get('title', '')}")
    print(f"  카테고리: {sample.get('metadata', {}).get('category', '')}")
    print(f"  본문(앞 100자): {sample.get('text', '')[:100]}...")


✅ 기존 데이터 사용 중 (10개 문서)
   재업로드하려면 FORCE_RELOAD = True 로 설정하세요.
✅ 벡터 스토어 준비 완료

[샘플 문서]
  제목: 
  카테고리: 
  본문(앞 100자): MongoDB Connection Pool 고갈 문제 해결 가이드

증상:
- 'connection pool exhausted' 또는 'too many connections' 오류...


## Step 7. Atlas 인덱스 자동 생성

pymongo `SearchIndexModel`을 사용해 **Vector Search 인덱스**와 **Atlas Search 인덱스**를  
코드에서 직접 생성하고, `READY` 상태까지 자동으로 대기합니다.

In [7]:
import time
from pymongo.operations import SearchIndexModel


def create_atlas_indexes(
    collection: pymongo.collection.Collection,
    force_recreate: bool = False,
) -> bool:
    """
    Vector Search 인덱스와 Atlas Search 인덱스를 자동으로 생성합니다.
    pymongo 4.7+의 SearchIndexModel / create_search_indexes() API를 사용합니다.

    Args:
        collection    : MongoDB 컬렉션
        force_recreate: True이면 기존 인덱스를 삭제하고 재생성

    Returns:
        새로 생성된 인덱스가 있으면 True
    """
    # 현재 존재하는 인덱스 목록 수집
    try:
        existing = {idx["name"] for idx in collection.list_search_indexes()}
    except Exception:
        existing = set()

    to_create = []

    # ── [1] Vector Search 인덱스 ──────────────────────────────────
    if VECTOR_INDEX_NAME in existing:
        if force_recreate:
            print(f"  기존 인덱스 삭제: {VECTOR_INDEX_NAME}")
            collection.drop_search_index(VECTOR_INDEX_NAME)
            _wait_for_drop(collection, VECTOR_INDEX_NAME)
        else:
            print(f"  ✅ 이미 존재: {VECTOR_INDEX_NAME}")

    if VECTOR_INDEX_NAME not in existing or force_recreate:
        to_create.append(
            SearchIndexModel(
                definition={
                    "fields": [
                        {
                            "type": "vector",
                            "path": EMBEDDING_FIELD,         # 임베딩 필드명
                            "numDimensions": EMBEDDING_DIMS, # voyage-4: 1024차원
                            "similarity": "cosine",          # 코사인 유사도
                        }
                    ]
                },
                name=VECTOR_INDEX_NAME,
                type="vectorSearch",   # ← Vector Search 타입 지정 필수
            )
        )

    # ── [2] Atlas Search (전문 검색) 인덱스 ──────────────────────
    if SEARCH_INDEX_NAME in existing:
        if force_recreate:
            print(f"  기존 인덱스 삭제: {SEARCH_INDEX_NAME}")
            collection.drop_search_index(SEARCH_INDEX_NAME)
            _wait_for_drop(collection, SEARCH_INDEX_NAME)
        else:
            print(f"  ✅ 이미 존재: {SEARCH_INDEX_NAME}")

    if SEARCH_INDEX_NAME not in existing or force_recreate:
        to_create.append(
            SearchIndexModel(
                definition={
                    "mappings": {
                        "dynamic": False,
                        "fields": {
                            TEXT_FIELD: {"type": "string"},     # 본문 키워드 검색
                            "metadata": {
                                "type": "document",
                                "fields": {
                                    "title": {"type": "string"}, # 제목 키워드 검색
                                },
                            },
                        },
                    }
                },
                name=SEARCH_INDEX_NAME,
                type="search",          # ← Atlas Search 타입 지정 필수
            )
        )

    # ── 생성 요청 ─────────────────────────────────────────────────
    if to_create:
        names = [m.document["name"] for m in to_create]
        print(f"  인덱스 생성 요청: {names}")
        collection.create_search_indexes(to_create)  # 비동기 생성
        return True

    return False


def _wait_for_drop(collection, index_name, timeout=120, interval=3):
    """인덱스 삭제 완료까지 대기 (내부 헬퍼)."""
    start = time.time()
    while time.time() - start < timeout:
        names = {idx["name"] for idx in collection.list_search_indexes()}
        if index_name not in names:
            return
        time.sleep(interval)


def wait_for_indexes_ready(
    collection: pymongo.collection.Collection,
    index_names: List[str],
    timeout: int = 300,
    poll_interval: int = 5,
) -> bool:
    """
    지정한 인덱스들이 모두 'READY' 상태가 될 때까지 폴링합니다.

    Args:
        collection   : MongoDB 컬렉션
        index_names  : 대기할 인덱스 이름 목록
        timeout      : 최대 대기 시간 (초, 기본 300초)
        poll_interval: 폴링 간격 (초, 기본 5초)

    Returns:
        모든 인덱스 READY이면 True, 타임아웃이면 False
    """
    target = set(index_names)
    start  = time.time()
    print(f"  인덱스 빌드 대기 중 (최대 {timeout}초)...")

    while time.time() - start < timeout:
        ready, parts = set(), []

        for idx in collection.list_search_indexes():
            if idx["name"] in target:
                status = idx.get("status", "UNKNOWN")
                parts.append(f"{idx['name']}: {status}")
                if status == "READY":
                    ready.add(idx["name"])

        elapsed = int(time.time() - start)
        print(f"  [{elapsed:>3}s] {' | '.join(parts)}", end="\r", flush=True)

        if ready == target:
            print(f"\n  ✅ 모든 인덱스 READY! ({elapsed}초 소요)")
            return True

        time.sleep(poll_interval)

    print(f"\n  ⚠️  타임아웃 ({timeout}초): 인덱스가 아직 준비되지 않았습니다.")
    return False


print("✅ 인덱스 생성 함수 정의 완료")

✅ 인덱스 생성 함수 정의 완료


In [8]:
# 인덱스 생성 실행
# force_recreate=True 로 설정하면 기존 인덱스를 삭제하고 재생성합니다
FORCE_RECREATE = False

print("[인덱스 생성]")
created = create_atlas_indexes(collection, force_recreate=FORCE_RECREATE)

if created:
    # 새로 생성된 경우 READY 상태까지 대기
    wait_for_indexes_ready(
        collection,
        index_names=[VECTOR_INDEX_NAME, SEARCH_INDEX_NAME],
        timeout=300,
        poll_interval=5,
    )
else:
    # 이미 모두 존재하는 경우 현재 상태만 출력
    print("\n[현재 인덱스 상태]")
    for idx in collection.list_search_indexes():
        print(f"  {idx['name']}: {idx.get('status', 'UNKNOWN')} (type: {idx.get('type', '-')})")

[인덱스 생성]
  ✅ 이미 존재: vector_index
  ✅ 이미 존재: search_index

[현재 인덱스 상태]
  search_index: READY (type: search)
  vector_index: READY (type: vectorSearch)


## Step 8. 하이브리드 검색 함수 구현

### 8-1. 벡터 검색 ($vectorSearch)

voyage-4 임베딩을 사용한 **시맨틱(의미 기반) 검색**

In [9]:
def vector_search(
    collection: pymongo.collection.Collection,
    query_embedding: List[float],
    k: int = 10,
) -> List[Dict[str, Any]]:
    """
    $vectorSearch를 사용한 시맨틱 검색.

    Args:
        collection     : MongoDB 컬렉션
        query_embedding: 쿼리의 임베딩 벡터 (voyage-4 생성)
        k              : 반환할 최대 결과 수

    Returns:
        검색 결과 목록 (vectorSearchScore 포함)
    """
    pipeline = [
        {
            "$vectorSearch": {
                "index": VECTOR_INDEX_NAME,       # Atlas 벡터 인덱스 이름
                "path": EMBEDDING_FIELD,           # 임베딩 필드명
                "queryVector": query_embedding,    # 쿼리 벡터
                "numCandidates": k * 10,           # 후보군 수 (k의 10배 권장)
                "limit": k,                        # 최종 반환 수
            }
        },
        {
            "$project": {
                "_id": 1,
                TEXT_FIELD: 1,
                "title": 1,
                "category": 1,
                "source": 1,
                "vector_score": {"$meta": "vectorSearchScore"},  # 코사인 유사도 점수
            }
        },
    ]
    return list(collection.aggregate(pipeline))


# 테스트: 벡터 검색
test_query = "connection pool 문제"
test_embedding = embeddings.embed_query(test_query)
v_results = vector_search(collection, test_embedding, k=5)

print(f"[벡터 검색] 쿼리: '{test_query}'")
print(f"결과 {len(v_results)}개:")
for i, r in enumerate(v_results, 1):
    meta = r.get("metadata", {})
    print(f"  {i}. [{meta.get('category', '-')}] {meta.get('title', '-')} "
          f"(score: {r.get('vector_score', 0):.4f})")

[벡터 검색] 쿼리: 'connection pool 문제'
결과 5개:
  1. [-] - (score: 0.7902)
  2. [-] - (score: 0.6874)
  3. [-] - (score: 0.6622)
  4. [-] - (score: 0.6524)
  5. [-] - (score: 0.6455)


### 8-2. 전문 검색 ($search)

Atlas Search를 사용한 **키워드 기반 검색**

In [10]:
def text_search(
    collection: pymongo.collection.Collection,
    query: str,
    k: int = 10,
) -> List[Dict[str, Any]]:
    """
    $search를 사용한 전문 검색(Full-Text Search).
    Atlas Search 인덱스(search_index)가 필요합니다.

    Args:
        collection: MongoDB 컬렉션
        query     : 검색 쿼리 문자열
        k         : 반환할 최대 결과 수

    Returns:
        검색 결과 목록 (searchScore 포함)
    """
    pipeline = [
        {
            "$search": {
                "index": SEARCH_INDEX_NAME,         # Atlas Search 인덱스 이름
                "text": {
                    "query": query,
                    "path": [TEXT_FIELD, "title"],  # 검색할 필드
                },
            }
        },
        {"$limit": k},
        {
            "$project": {
                "_id": 1,
                TEXT_FIELD: 1,
                "title": 1,
                "category": 1,
                "source": 1,
                "text_score": {"$meta": "searchScore"},  # Atlas Search 관련도 점수
            }
        },
    ]
    return list(collection.aggregate(pipeline))


# 테스트: 전문 검색
t_results = text_search(collection, test_query, k=5)

print(f"[전문 검색] 쿼리: '{test_query}'")
print(f"결과 {len(t_results)}개:")
for i, r in enumerate(t_results, 1):
    meta = r.get("metadata", {})
    print(f"  {i}. [{meta.get('category', '-')}] {meta.get('title', '-')} "
          f"(score: {r.get('text_score', 0):.4f})")

[전문 검색] 쿼리: 'connection pool 문제'
결과 2개:
  1. [-] - (score: 2.9765)
  2. [-] - (score: 0.7325)


### 8-3. RRF (Reciprocal Rank Fusion)

두 검색 결과를 **RRF 공식**으로 융합합니다.

$$\text{RRF\_score}(d) = \sum_{i \in \text{검색시스템}} \frac{1}{k + \text{rank}_i(d)}$$

- $k = 60$ (기본 상수, 높을수록 순위 차이 효과 완화)
- $\text{rank}_i$ : $i$번째 검색 시스템에서 문서의 순위 (1부터 시작)

In [11]:
def reciprocal_rank_fusion(
    vector_results: List[Dict],
    text_results: List[Dict],
    rrf_k: int = 60,
) -> List[Dict[str, Any]]:
    """
    RRF(Reciprocal Rank Fusion)으로 벡터 검색과 전문 검색 결과를 융합합니다.

    RRF 공식:
        RRF_score(d) = Σ  1 / (k + rank_i(d))

    Args:
        vector_results: 벡터 검색 결과 (순위 포함)
        text_results  : 전문 검색 결과 (순위 포함)
        rrf_k         : RRF 상수 k (기본값 60)

    Returns:
        RRF 점수로 정렬된 융합 결과 목록
    """
    rrf_map: Dict[str, Dict] = {}

    # 벡터 검색 결과 처리
    for rank, doc in enumerate(vector_results, start=1):
        doc_id = str(doc["_id"])
        if doc_id not in rrf_map:
            rrf_map[doc_id] = {
                "doc": doc, "rrf_score": 0.0,
                "vector_rank": None, "text_rank": None,
                "vector_score": None, "text_score": None,
            }
        rrf_map[doc_id]["rrf_score"]   += 1.0 / (rrf_k + rank)  # RRF 공식
        rrf_map[doc_id]["vector_rank"]  = rank
        rrf_map[doc_id]["vector_score"] = doc.get("vector_score")

    # 전문 검색 결과 처리
    for rank, doc in enumerate(text_results, start=1):
        doc_id = str(doc["_id"])
        if doc_id not in rrf_map:
            rrf_map[doc_id] = {
                "doc": doc, "rrf_score": 0.0,
                "vector_rank": None, "text_rank": None,
                "vector_score": None, "text_score": None,
            }
        rrf_map[doc_id]["rrf_score"]  += 1.0 / (rrf_k + rank)   # RRF 공식
        rrf_map[doc_id]["text_rank"]   = rank
        rrf_map[doc_id]["text_score"]  = doc.get("text_score")

    # RRF 점수 기준 내림차순 정렬
    return sorted(rrf_map.values(), key=lambda x: x["rrf_score"], reverse=True)


# 테스트: RRF 융합
rrf_results = reciprocal_rank_fusion(v_results, t_results, rrf_k=60)

print(f"[RRF 융합 결과] 쿼리: '{test_query}'")
print(f"{'순위':<4} {'제목':<35} {'벡터순위':<8} {'텍스트순위':<10} RRF점수")
print("-" * 75)
for i, r in enumerate(rrf_results, 1):
    # langchain-mongodb는 metadata 필드를 최상위 레벨에 저장합니다
    title  = (r["doc"].get("title") or "제목없음")[:33]
    v_rank = str(r["vector_rank"]) if r["vector_rank"] else "-"
    t_rank = str(r["text_rank"])   if r["text_rank"]   else "-"
    print(f"{i:<4} {title:<35} {v_rank:<8} {t_rank:<10} {r['rrf_score']:.6f}")


[RRF 융합 결과] 쿼리: 'connection pool 문제'
순위   제목                                  벡터순위     텍스트순위      RRF점수
---------------------------------------------------------------------------
1    제목없음                                1        1          0.032787
2    제목없음                                5        2          0.031514
3    제목없음                                2        -          0.016129
4    제목없음                                3        -          0.015873
5    제목없음                                4        -          0.015625


### 8-4. 하이브리드 검색 통합 함수

In [12]:
def hybrid_search(
    collection: pymongo.collection.Collection,
    query: str,
    embeddings: VoyageAIEmbeddings,
    k: int = 10,
    rrf_k: int = 60,
    verbose: bool = True,
) -> List[Dict]:
    """
    하이브리드 검색 = 벡터 검색 + 전문 검색 + RRF 융합.

    Args:
        collection: MongoDB 컬렉션
        query     : 검색 쿼리
        embeddings: Voyage AI 임베딩 모델
        k         : 각 검색에서 가져올 결과 수
        rrf_k     : RRF 상수 k
        verbose   : 결과 상세 출력 여부

    Returns:
        RRF로 융합된 검색 결과 목록
    """
    # 1. 쿼리 임베딩 생성
    query_embedding = embeddings.embed_query(query)

    # 2. 벡터 검색
    v_results = vector_search(collection, query_embedding, k=k)

    # 3. 전문 검색
    t_results = text_search(collection, query, k=k)

    # 4. RRF 융합
    rrf_results = reciprocal_rank_fusion(v_results, t_results, rrf_k=rrf_k)

    if verbose:
        print(f"\n벡터 검색: {len(v_results)}개, 전문 검색: {len(t_results)}개 → RRF 융합: {len(rrf_results)}개")
        print(f"\n{'순위':<4} {'제목':<35} {'벡터순위':<8} {'텍스트순위':<10} {'RRF점수':<12} 카테고리")
        print("-" * 80)
        for i, r in enumerate(rrf_results[:5], 1):
            # langchain-mongodb는 metadata 필드를 최상위 레벨에 저장합니다
            title  = (r["doc"].get("title") or "제목없음")[:33]
            cat    = (r["doc"].get("category") or "-")
            v_rank = str(r["vector_rank"]) if r["vector_rank"] else "-"
            t_rank = str(r["text_rank"])   if r["text_rank"]   else "-"
            print(f"{i:<4} {title:<35} {v_rank:<8} {t_rank:<10} {r['rrf_score']:<12.6f} {cat}")

    return rrf_results


# 테스트
print("=" * 80)
print("하이브리드 검색 테스트")
print("=" * 80)
results = hybrid_search(collection, "MongoDB 메모리 부족 문제", embeddings, k=10, verbose=True)


하이브리드 검색 테스트

벡터 검색: 10개, 전문 검색: 9개 → RRF 융합: 10개

순위   제목                                  벡터순위     텍스트순위      RRF점수        카테고리
--------------------------------------------------------------------------------
1    제목없음                                1        2          0.032522     -
2    제목없음                                2        1          0.032522     -
3    제목없음                                3        5          0.031258     -
4    제목없음                                4        4          0.031250     -
5    제목없음                                6        3          0.031025     -


## Step 9. RAG 체인 구성

LangChain LCEL(LangChain Expression Language)로 RAG 파이프라인을 구성합니다.

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def format_context(rrf_results: List[Dict], top_k: int = 3) -> str:
    """RRF 결과 상위 문서로 RAG 컨텍스트를 구성합니다."""
    context_parts = []
    for i, result in enumerate(rrf_results[:top_k], start=1):
        doc     = result["doc"]
        meta    = doc.get("metadata", {})
        title   = doc.get("title", "")
        content = doc.get(TEXT_FIELD, "")
        rrf_sc  = result["rrf_score"]
        context_parts.append(
            f"--- 문서 {i}: {title} (RRF 점수: {rrf_sc:.4f}) ---\n{content}"
        )
    return "\n\n".join(context_parts)


# RAG 프롬프트 템플릿
rag_prompt = ChatPromptTemplate.from_template(
    """당신은 MongoDB 전문가입니다.
주어진 컨텍스트를 바탕으로 MongoDB 장애와 관련된 질문에 정확하고 실용적인 답변을 제공하세요.

[참고 문서]
{context}

[질문]
{question}

[답변 지침]
1. 참고 문서의 정보를 우선적으로 활용하세요.
2. 구체적인 해결 방법과 명령어를 포함하세요.
3. 단계적으로 설명하세요 (즉각 조치 → 근본 원인 해결 → 예방).
4. 참고 문서에 없는 정보는 일반적인 MongoDB 지식으로 보완하세요.

[답변]
"""
)

# LCEL 체인: 프롬프트 → LLM → 문자열 파서
rag_chain = rag_prompt | llm | StrOutputParser()

print("✅ RAG 체인 구성 완료")
print("   구조: ChatPromptTemplate → ChatOpenAI → StrOutputParser")

✅ RAG 체인 구성 완료
   구조: ChatPromptTemplate → ChatOpenAI → StrOutputParser


## Step 10. 통합 Q&A 함수

In [14]:
def ask_mongodb_question(
    question: str,
    collection: pymongo.collection.Collection,
    embeddings: VoyageAIEmbeddings,
    llm: ChatOpenAI,
    context_top_k: int = 3,
    search_k: int = 10,
    rrf_k: int = 60,
    verbose: bool = True,
) -> str:
    """
    MongoDB 장애 관련 질문에 RAG 기반으로 답변합니다.

    흐름:
        질문 → 하이브리드 검색(벡터+전문+RRF) → 컨텍스트 구성 → LLM 답변

    Args:
        question     : 사용자 질문
        collection   : MongoDB 컬렉션
        embeddings   : Voyage AI 임베딩
        llm          : OpenAI LLM
        context_top_k: RAG 컨텍스트에 포함할 상위 문서 수
        search_k     : 각 검색에서 가져올 결과 수
        rrf_k        : RRF 상수 k
        verbose      : RRF 결과 상세 출력 여부

    Returns:
        생성된 답변 문자열
    """
    print(f"\n{'='*80}")
    print(f"  질문: {question}")
    print(f"{'='*80}")

    # 1. 하이브리드 검색
    print("\n[1] 하이브리드 검색 실행...")
    rrf_results = hybrid_search(
        collection=collection,
        query=question,
        embeddings=embeddings,
        k=search_k,
        rrf_k=rrf_k,
        verbose=verbose,
    )

    # 2. 컨텍스트 구성
    print(f"\n[2] 상위 {context_top_k}개 문서로 컨텍스트 구성")
    context = format_context(rrf_results, top_k=context_top_k)

    # 3. LLM 답변 생성
    print("\n[3] LLM 답변 생성 중...")
    answer = rag_chain.invoke({"context": context, "question": question})

    print(f"\n{'─'*80}")
    print("  [답변]")
    print(f"{'─'*80}")
    print(answer)
    print(f"{'='*80}")

    return answer


print("✅ Q&A 함수 정의 완료")

✅ Q&A 함수 정의 완료


## Step 11. Q&A 실행 예제

In [15]:
# 예제 1: Connection Pool 문제
answer1 = ask_mongodb_question(
    question="MongoDB connection pool이 고갈되었을 때 어떻게 해결하나요?",
    collection=collection,
    embeddings=embeddings,
    llm=llm,
    context_top_k=3,
    verbose=True,
)


  질문: MongoDB connection pool이 고갈되었을 때 어떻게 해결하나요?

[1] 하이브리드 검색 실행...

벡터 검색: 10개, 전문 검색: 8개 → RRF 융합: 10개

순위   제목                                  벡터순위     텍스트순위      RRF점수        카테고리
--------------------------------------------------------------------------------
1    제목없음                                1        1          0.032787     -
2    제목없음                                2        4          0.031754     -
3    제목없음                                4        2          0.031754     -
4    제목없음                                3        5          0.031258     -
5    제목없음                                9        3          0.030366     -

[2] 상위 3개 문서로 컨텍스트 구성

[3] LLM 답변 생성 중...

────────────────────────────────────────────────────────────────────────────────
  [답변]
────────────────────────────────────────────────────────────────────────────────
MongoDB connection pool이 고갈되었을 때 해결하는 방법은 다음과 같습니다.

### 1. 즉각 조치
먼저, 현재 커넥션 상태를 확인하여 문제의 심각성을 파악합니다.

```javascript
db.serverStatus().con

In [16]:
# 예제 2: 복제 지연 문제
answer2 = ask_mongodb_question(
    question="Replication lag이 발생했을 때 원인과 해결 방법은?",
    collection=collection,
    embeddings=embeddings,
    llm=llm,
    context_top_k=3,
    verbose=True,
)


  질문: Replication lag이 발생했을 때 원인과 해결 방법은?

[1] 하이브리드 검색 실행...

벡터 검색: 10개, 전문 검색: 9개 → RRF 융합: 10개

순위   제목                                  벡터순위     텍스트순위      RRF점수        카테고리
--------------------------------------------------------------------------------
1    제목없음                                1        1          0.032787     -
2    제목없음                                3        3          0.031746     -
3    제목없음                                5        2          0.031514     -
4    제목없음                                2        6          0.031281     -
5    제목없음                                7        4          0.030550     -

[2] 상위 3개 문서로 컨텍스트 구성

[3] LLM 답변 생성 중...

────────────────────────────────────────────────────────────────────────────────
  [답변]
────────────────────────────────────────────────────────────────────────────────
Replication lag이 발생했을 때의 원인과 해결 방법은 다음과 같습니다.

### 원인
1. **Primary에 과도한 쓰기 부하**: Primary 서버에 많은 쓰기 작업이 발생하면 Secondary 서버가 이를 따라잡지 못해 지연이 발생할 수 있습니

In [ ]:
# 예제 3: 쿼리 성능 문제
answer3 = ask_mongodb_question(
    question="쿼리가 너무 느릴 때 어떻게 최적화하나요?",
    collection=collection,
    embeddings=embeddings,
    llm=llm,
    context_top_k=3,
    verbose=True,
)

In [ ]:
# 예제 4: 디스크 공간 부족
answer4 = ask_mongodb_question(
    question="MongoDB 서버의 디스크 공간이 부족할 때 어떻게 대처하나요?",
    collection=collection,
    embeddings=embeddings,
    llm=llm,
    context_top_k=3,
    verbose=True,
)

## Step 12. RRF 파라미터 실험

RRF 상수 `k` 값에 따른 결과 변화를 비교합니다.

In [ ]:
query = "MongoDB 메모리 사용량이 높을 때 해결 방법"
query_embedding = embeddings.embed_query(query)

# 각 검색 결과
v_res = vector_search(collection, query_embedding, k=10)
t_res = text_search(collection, query, k=10)

print(f"쿼리: '{query}'")
print("=" * 80)

# k 값별 RRF 결과 비교
for k_val in [10, 60, 100]:
    rrf_res = reciprocal_rank_fusion(v_res, t_res, rrf_k=k_val)
    print(f"\n[RRF k={k_val}]")
    print(f"  {'순위':<4} {'제목':<35} {'벡터순위':<8} {'텍스트순위':<10} RRF점수")
    print("  " + "-" * 70)
    for i, r in enumerate(rrf_res[:3], 1):
        # langchain-mongodb는 metadata 필드를 최상위 레벨에 저장합니다
        title = (r["doc"].get("title") or "제목없음")[:33]
        v_r   = str(r["vector_rank"]) if r["vector_rank"] else "-"
        t_r   = str(r["text_rank"])   if r["text_rank"]   else "-"
        print(f"  {i:<4} {title:<35} {v_r:<8} {t_r:<10} {r['rrf_score']:.6f}")


## Step 13. 자유 질문

MongoDB 장애에 관한 질문을 직접 입력해보세요.

In [ ]:
# 여기에 질문을 입력하세요
my_question = "MongoDB Atlas에서 Primary가 선출되지 않을 때 어떻게 해야 하나요?"

answer = ask_mongodb_question(
    question=my_question,
    collection=collection,
    embeddings=embeddings,
    llm=llm,
    context_top_k=3,
    verbose=True,
)

## Step 14. 리소스 정리

In [ ]:
# MongoDB 연결 종료
client.close()
print("✅ MongoDB 연결 종료")

---

## 정리

이 노트북에서 구현한 내용:

| 단계 | 내용 |
|------|------|
| **문서 준비** | MongoDB 장애 대응 지식 베이스 10가지 시나리오 |
| **임베딩** | Voyage AI `voyage-4` (1024차원) |
| **저장** | MongoDB Atlas Vector Search |
| **인덱스 자동 생성** | `SearchIndexModel` + `create_search_indexes()` (pymongo 4.7+) |
| **벡터 검색** | `$vectorSearch` (코사인 유사도) |
| **전문 검색** | `$search` Atlas Search |
| **RRF 융합** | `1/(k+rank)` 공식으로 두 결과를 통합 |
| **답변 생성** | OpenAI `gpt-4o-mini` + LangChain LCEL |

### 다음 단계로 확장하기

- **문서 추가**: `MONGODB_KNOWLEDGE_BASE`에 더 많은 장애 시나리오 추가
- **메타데이터 필터링**: `$vectorSearch`의 `filter` 옵션으로 카테고리별 검색
- **멀티턴 대화**: LangChain `ConversationBufferMemory`로 대화 이력 관리
- **스트리밍 응답**: `rag_chain.stream()` 사용으로 실시간 출력
- **평가**: RAGAS 라이브러리로 RAG 품질 측정